In [12]:
import tkinter as tk
from tkinter import ttk
import copy
import random

# =====================================================================
# CẤU HÌNH DỮ LIỆU ĐỒ THỊ BẢN ĐỒ (CSP)
# =====================================================================
neighbors = {
    'PhuNhuan': ['BinhThanh', 'Quan3'],
    'BinhThanh': ['PhuNhuan', 'Quan3', 'Quan1'],
    'Quan3': ['PhuNhuan', 'BinhThanh', 'Quan1', 'Quan10'],
    'Quan1': ['BinhThanh', 'Quan3', 'Quan4', 'Quan5'],
    'Quan10': ['Quan3', 'Quan5'],
    'Quan4': ['Quan1', 'Quan5'],
    'Quan5': ['Quan1', 'Quan4', 'Quan10']
}

# Thứ tự duyệt các quận
districts = ['Quan1', 'Quan3', 'Quan5', 'BinhThanh', 'PhuNhuan', 'Quan10', 'Quan4']

# Tên hiển thị trên giao diện
display_names = {
    'PhuNhuan': 'Phú Nhuận', 'BinhThanh': 'Bình Thạnh', 'Quan3': 'Quận 3',
    'Quan1': 'Quận 1', 'Quan10': 'Quận 10', 'Quan4': 'Quận 4', 'Quan5': 'Quận 5'
}

# Tọa độ vẽ UI
positions = {
    'PhuNhuan': (200, 100), 'BinhThanh': (380, 100), 'Quan3': (160, 240),
    'Quan1': (320, 260), 'Quan10': (60, 320), 'Quan5': (150, 420), 'Quan4': (350, 420)
}

# Miền giá trị màu sắc
color_map = {
    0: {"hex": "#E0E0E0", "name": "Chưa tô"},
    1: {"hex": "#FF4B4B", "name": "Đỏ"},
    2: {"hex": "#4BFF4B", "name": "Xanh lá"},
    3: {"hex": "#4B4BFF", "name": "Xanh dương"},
    4: {"hex": "#FFD700", "name": "Vàng"}
}


# =====================================================================
# KHỞI TẠO ỨNG DỤNG VÀ GIAO DIỆN CHÍNH
# =====================================================================
class MultiAlgoMapColoringApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Visual Tô Bản Đồ TP.HCM - Đồ Án UTEX")
        self.root.geometry("600x720")
        self.root.configure(bg="#F8F9FA")

        self.radius = 45
        self.history = []  # Ghi lại lịch sử các bước để chạy hoạt họa
        self.anim_index = 0

        # --- UI: Thanh chọn thuật toán ---
        self.lbl_select = tk.Label(root, text="Chọn thuật toán tô màu:", font=("Arial", 11, "bold"), bg="#F8F9FA", fg="#333")
        self.lbl_select.pack(pady=(10, 2))

        self.algo_var = tk.StringVar()
        self.combobox_algo = ttk.Combobox(root, textvariable=self.algo_var, font=("Arial", 11), state="readonly", width=35)
        self.combobox_algo['values'] = (
            "1. Backtracking",
            "2. Thuật toán Forward Checking",
            "3. Backtracking kết hợp AC-3",
            "4. Thuật toán Min-Conflicts"
        )
        self.combobox_algo.current(0)
        self.combobox_algo.pack(pady=5)

        # --- UI: Nút bấm và Trạng thái ---
        self.btn_start = tk.Button(root, text="⚡ Bắt đầu chạy", font=("Arial", 11, "bold"),
                                   bg="#1A73E8", fg="white", bd=0, padx=20, pady=8, cursor="hand2")
        self.btn_start.pack(pady=10)
        self.btn_start.config(command=self.start_processing)

        self.lbl_status = tk.Label(root, text="Chọn thuật toán và nhấn nút để xem quá trình chạy...",
                                   font=("Arial", 11, "italic"), bg="#F8F9FA", fg="#5F6368")
        self.lbl_status.pack(pady=2)

        # --- UI: Khung vẽ (Canvas) ---
        self.canvas = tk.Canvas(root, width=550, height=480, bg="white", bd=1, relief="solid")
        self.canvas.pack(pady=10)

        self.draw_base_graph()

    def draw_base_graph(self):
        """Vẽ lại đồ thị trắng chưa tô màu"""
        self.canvas.delete("all")
        drawn_edges = set()
        for node, edges in neighbors.items():
            for neighbor in edges:
                edge = tuple(sorted((node, neighbor)))
                if edge not in drawn_edges:
                    x1, y1 = positions[node]
                    x2, y2 = positions[neighbor]
                    self.canvas.create_line(x1, y1, x2, y2, fill="#A0A0A0", width=3)
                    drawn_edges.add(edge)

        for dist in positions.keys():
            x, y = positions[dist]
            self.canvas.create_oval(x - self.radius, y - self.radius, x + self.radius, y + self.radius,
                                    fill="#E0E0E0", outline="#5F6368", width=2, tags=dist)
            self.canvas.create_text(x, y, text=display_names[dist], font=("Arial", 10, "bold"),
                                    fill="black", tags=f"text_{dist}")

    # =====================================================================
    # THUẬT TOÁN 1: BACKTRACKING
    # =====================================================================
    def is_valid_pure_bt(self, node, color, current_assign):
        for neighbor in neighbors[node]:
            if current_assign.get(neighbor) == color:
                return False
        return True

    def solve_pure_backtracking(self, index, current_assign):
        if index == len(districts): return True
        node = districts[index]
        name_vn = display_names[node]

        for color_id in [1, 2, 3, 4]:
            self.history.append((node, color_id, f"[BT] Thử màu {color_map[color_id]['name']} cho {name_vn}..."))

            if self.is_valid_pure_bt(node, color_id, current_assign):
                current_assign[node] = color_id
                self.history.append((node, color_id, f"✅ [BT] {name_vn} hợp lệ."))

                if self.solve_pure_backtracking(index + 1, current_assign): return True

                del current_assign[node]
                self.history.append((node, 0, f"❌ [BT] Bị kẹt nhánh sau! Quay lui xóa màu tại {name_vn}..."))
            else:
                self.history.append((node, color_id, f"⚠️ [BT] Trùng màu lân cận! Bỏ màu ở {name_vn}..."))
        return False

    # =====================================================================
    # THUẬT TOÁN 2: FORWARD CHECKING
    # =====================================================================
    def update_domains_fc(self, assigned_node, assigned_color, domains, current_assign):
        for neighbor in neighbors[assigned_node]:
            if neighbor not in current_assign:
                if assigned_color in domains[neighbor]:
                    domains[neighbor].remove(assigned_color)
                    if len(domains[neighbor]) == 0:
                        return False
        return True

    def solve_forward_checking(self, index, current_assign, domains):
        if index == len(districts): return True
        node = districts[index]
        name_vn = display_names[node]

        for color_id in list(domains[node]):
            self.history.append((node, color_id, f"[FC] Thử màu {color_map[color_id]['name']} cho {name_vn}..."))

            domains_copy = copy.deepcopy(domains)
            domains_copy[node] = [color_id]
            current_assign[node] = color_id

            if self.update_domains_fc(node, color_id, domains_copy, current_assign):
                self.history.append((node, color_id, f"✅ [FC] {name_vn} hợp lệ (Hàng xóm an toàn)."))
                if self.solve_forward_checking(index + 1, current_assign, domains_copy): return True
                self.history.append((node, 0, f"❌ [FC] Kẹt ở nhánh sau! Quay lui tại {name_vn}..."))
            else:
                self.history.append((node, color_id, f"⚠️ [FC] Hàng xóm bị cạn màu (Wipeout)! Bỏ thử nghiệm..."))

            if node in current_assign: del current_assign[node]
        return False

    # =====================================================================
    # THUẬT TOÁN 3: AC-3
    # =====================================================================
    def remove_inconsistent_values(self, Xi, Xj, domains):
        removed = False
        for x in list(domains[Xi]):
            if len(domains[Xj]) == 1 and domains[Xj][0] == x:
                domains[Xi].remove(x)
                removed = True
        return removed

    def run_ac3_filter(self, domains, queue):
        while queue:
            Xi, Xj = queue.pop(0)
            if self.remove_inconsistent_values(Xi, Xj, domains):
                if not domains[Xi]: return False
                for Xk in neighbors[Xi]:
                    if Xk != Xj:
                        queue.append((Xk, Xi))
        return True

    def solve_ac3_mac(self, index, current_assign, domains):
        if index == len(districts): return True
        node = districts[index]
        name_vn = display_names[node]

        for color_id in list(domains[node]):
            self.history.append((node, color_id, f"[AC3] Thử màu {color_map[color_id]['name']} cho {name_vn}..."))

            domains_copy = copy.deepcopy(domains)
            domains_copy[node] = [color_id]
            current_assign[node] = color_id

            queue = [(neighbor, node) for neighbor in neighbors[node]]

            if self.run_ac3_filter(domains_copy, queue):
                self.history.append((node, color_id, f"✅ [AC3] {name_vn} hợp lệ (Qua bộ lọc)."))
                if self.solve_ac3_mac(index + 1, current_assign, domains_copy): return True
                self.history.append((node, 0, f"❌ [AC3] Nhánh sau vô nghiệm! Quay lui tại {name_vn}..."))
            else:
                self.history.append((node, color_id, f"⚠️ [AC3] Xung đột sớm! Bỏ màu ở {name_vn}..."))

            if node in current_assign: del current_assign[node]
        return False

    # =====================================================================
    # THUẬT TOÁN 4: MIN-CONFLICTS
    # =====================================================================
    def count_conflicts(self, node, color, assignment):
        """Hàm đếm số lượng vi phạm ràng buộc của một quận với màu cụ thể"""
        count = 0
        for neighbor in neighbors[node]:
            if assignment.get(neighbor) == color:
                count += 1
        return count

    def solve_min_conflicts(self, max_steps=100):
        current_assign = {node: random.choice([1, 2, 3, 4]) for node in districts}

        for node, color in current_assign.items():
            self.history.append((node, color, f"[MC] Khởi tạo ngẫu nhiên {display_names[node]}..."))

        for step in range(max_steps):
            conflicted_vars = []
            for node in districts:
                if self.count_conflicts(node, current_assign[node], current_assign) > 0:
                    conflicted_vars.append(node)

            if not conflicted_vars:
                return True

            var = random.choice(conflicted_vars)
            name_vn = display_names[var]

            min_conflict_count = float('inf')
            best_colors = []

            for color_id in [1, 2, 3, 4]:
                conflicts = self.count_conflicts(var, color_id, current_assign)
                if conflicts < min_conflict_count:
                    min_conflict_count = conflicts
                    best_colors = [color_id]
                elif conflicts == min_conflict_count:
                    best_colors.append(color_id)

            chosen_color = random.choice(best_colors)
            current_assign[var] = chosen_color

            self.history.append((var, chosen_color, f"🔄 [MC] Sửa {name_vn} thành {color_map[chosen_color]['name']} (Xung đột giảm còn: {min_conflict_count})"))

        return False

    # =====================================================================
    # ĐIỀU PHỐI VÀ HOẠT HỌA GIAO DIỆN
    # =====================================================================
    def start_processing(self):
        """Khóa giao diện, chọn thuật toán và bắt đầu chạy"""
        self.btn_start.config(state="disabled")
        self.combobox_algo.config(state="disabled")
        self.draw_base_graph()

        self.history = []
        self.anim_index = 0
        selected_algo = self.algo_var.get()
        success = False

        if selected_algo == "1. Backtracking":
            success = self.solve_pure_backtracking(0, {})

        elif selected_algo == "2. Thuật toán Forward Checking":
            initial_domains = {dist: [1, 2, 3, 4] for dist in districts}
            success = self.solve_ac3_mac(0, {}, initial_domains)

        elif selected_algo == "3. Backtracking kết hợp AC-3":
            initial_domains = {dist: [1, 2, 3, 4] for dist in districts}
            success = self.solve_forward_checking(0, {}, initial_domains)

        elif selected_algo == "4. Thuật toán Min-Conflicts":
            success = self.solve_min_conflicts(max_steps=10)

        if success:
            self.history.append((None, None, f"🎉 THÀNH CÔNG! Giải xong bằng {selected_algo[:25]}..."))
        else:
            self.history.append((None, None, f"❌ KHÔNG TÌM THẤY LỜI GIẢI."))

        self.play_animation()

    def play_animation(self):
        """Chạy từng khung hình dựa trên mảng history đã ghi lại"""
        if self.anim_index >= len(self.history):
            self.btn_start.config(state="normal")
            self.combobox_algo.config(state="readonly")
            return

        node, color_id, status_text = self.history[self.anim_index]

        if "✅" in status_text or "🎉" in status_text:
            self.lbl_status.config(text=status_text, fg="green" if "✅" in status_text else "#1A73E8")
        elif "❌" in status_text or "⚠️" in status_text:
            self.lbl_status.config(text=status_text, fg="#D32F2F")
        else:
            self.lbl_status.config(text=status_text, fg="#333333")

        if node and color_id is not None:
            bg_color = color_map[color_id]["hex"]
            text_color = "black" if color_id in [0, 2, 4] else "white"
            self.canvas.itemconfig(node, fill=bg_color)
            self.canvas.itemconfig(f"text_{node}", fill=text_color)

        self.anim_index += 1

        self.root.after(400, self.play_animation)


if __name__ == "__main__":
    root = tk.Tk()
    app = MultiAlgoMapColoringApp(root)
    root.mainloop()